# Etapa 6 — Lógica Fuzzy, Integração Final e Validação com 2025

O sistema Mamdani (scikit-fuzzy) recebe:
- **presença** = P(facção | porte, fronteira, sintomas), vinda da rede bayesiana (Etapa 5);
- **tendência** = variação relativa dos homicídios de 2020–21 para 2022–23,

e devolve a **prioridade de monitoramento / inteligência policial** (0–100).

**Regras**
1. SE presença é alta E (tendência é alta OU estável) ENTÃO prioridade é alta.
2. SE presença é alta E tendência é queda ENTÃO prioridade é média (possível **domínio consolidado**).
3. SE presença é média E tendência é alta ENTÃO prioridade é alta (possível **expansão**).
4. SE presença é média E (tendência é estável OU queda) ENTÃO prioridade é média.
5. SE presença é baixa ENTÃO prioridade é baixa.

Depois conferimos a decisão com dois gabaritos: a presença em **2024** (o rótulo usado no treino) e as **entradas em 2025** (4ª edição do Cartografias), que o modelo nunca viu.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 95)


In [2]:
from fuzzy import criar_sistema, avaliar, rotulo
probs = pd.read_csv("dados/probabilidades_bayesianas.csv")
sistema, (presenca, tendencia, prioridade) = criar_sistema()
fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for k, var in enumerate([presenca, tendencia, prioridade]):
    for nome, termo in var.terms.items():
        ax[k].plot(var.universe, termo.mf, label=nome)
    ax[k].set_title(var.label); ax[k].legend(fontsize=7)
plt.tight_layout(); plt.savefig("figuras/etapa6_pertinencia.png", dpi=150)

In [3]:
probs["tend_cap"] = probs["variacao_hom"].clip(-1, 2)
probs["prioridade"] = [avaliar(sistema, p, t) for p, t in zip(probs["p_faccao"], probs["tend_cap"])]
probs["decisao_fuzzy"] = probs["prioridade"].map(rotulo)
probs["decisao_rigida"] = np.where(probs["p_faccao"] >= 0.5, "alta", "baixa")
print(probs["decisao_fuzzy"].value_counts()); print(probs["decisao_rigida"].value_counts())

decisao_fuzzy
baixa    406
alta     238
média    164
Name: count, dtype: int64
decisao_rigida
baixa    573
alta     235
Name: count, dtype: int64


## 1. Decisão fuzzy × decisão rígida, conferidas com o Cartografias 2024

In [4]:
def resumo(col):
    g = probs.groupby(col)
    return pd.DataFrame({"municípios": g.size(), "com facção 2024": g["ALVO"].mean(),
                         "em disputa 2024": g["REGIME"].apply(lambda s: (s == 2).mean())})
print("Decisão fuzzy:"); print(resumo("decisao_fuzzy").reindex(["baixa", "média", "alta"]).round(3))
print("\nDecisão rígida (P >= 0,5):"); print(resumo("decisao_rigida").round(3))

Decisão fuzzy:
               municípios  com facção 2024  em disputa 2024
decisao_fuzzy                                              
baixa                 406            0.128            0.017
média                 164            0.366            0.098
alta                  238            0.618            0.261

Decisão rígida (P >= 0,5):
                municípios  com facção 2024  em disputa 2024
decisao_rigida                                              
alta                   235            0.643            0.277
baixa                  573            0.188            0.035


## 2. Validação fora do tempo: o alerta antecipa a expansão de 2025?

Olhamos só os municípios **sem facção em 2024** e conferimos quantos passaram a ter facção em **2025**, segundo a 4ª edição do Cartografias.

In [5]:
from sklearn.metrics import roc_auc_score
sem = probs[probs["ALVO"] == 0].copy()
print(f"Municípios sem facção em 2024: {len(sem)}  |  passaram a ter em 2025: {int(sem['ALVO_2025'].sum())} ({sem['ALVO_2025'].mean():.1%})")
print("\nEntrada em 2025 por prioridade fuzzy (calculada só com dados até 2023):")
print(sem.groupby("decisao_fuzzy")["ALVO_2025"].agg(municipios="size", entraram="mean").reindex(["baixa", "média", "alta"]).round(3))
print("\nEntrada em 2025 pela decisão rígida:")
print(sem.groupby("decisao_rigida")["ALVO_2025"].agg(municipios="size", entraram="mean").round(3))
print(f"\nAUC da probabilidade bayesiana para prever a entrada em 2025: {roc_auc_score(sem['ALVO_2025'], sem['p_faccao']):.3f}")
print(f"AUC da prioridade fuzzy:                                      {roc_auc_score(sem['ALVO_2025'], sem['prioridade']):.3f}")

Municípios sem facção em 2024: 549  |  passaram a ter em 2025: 129 (23.5%)

Entrada em 2025 por prioridade fuzzy (calculada só com dados até 2023):
               municipios  entraram
decisao_fuzzy                      
baixa                 354     0.127
média                 104     0.413
alta                   91     0.451

Entrada em 2025 pela decisão rígida:
                municipios  entraram
decisao_rigida                      
alta                    84     0.536
baixa                  465     0.181

AUC da probabilidade bayesiana para prever a entrada em 2025: 0.765
AUC da prioridade fuzzy:                                      0.754


In [6]:
pp = np.linspace(0, 1, 101)
plt.figure(figsize=(6.5, 3.8))
for t, nome in [(-0.5, "homicídios caindo"), (0, "estável"), (1, "homicídios dobrando")]:
    plt.plot(pp, [avaliar(sistema, p, t) for p in pp], label=f"tendência: {nome}")
plt.plot(pp, np.where(pp >= 0.5, 100, 0), "k--", lw=1, label="limiar rígido 0,5")
plt.xlabel("P(facção) da rede bayesiana"); plt.ylabel("prioridade (0–100)")
plt.title("Saída fuzzy suave × limiar rígido"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figuras/etapa6_fuzzy_vs_rigido.png", dpi=150)

In [7]:
# Municípios sem facção em 2024 com prioridade fuzzy alta: candidatos à expansão
cand = sem.sort_values("prioridade", ascending=False).head(15)
cand[["uf", "municipio", "pop_2022", "p_faccao", "variacao_hom", "prioridade", "ALVO_2025"]].round(3)

,uf,municipio,pop_2022,p_faccao,variacao_hom,prioridade,ALVO_2025
734,TO,LAGOA DA CONFUSÃO,15288.0,0.407,1.857,83.254,0
539,PA,ORIXIMINÁ,68294.0,0.914,0.040,82.978,0
624,RO,JARU,50591.0,0.854,0.167,81.944,1
38,AM,CARAUARI,28742.0,0.383,1.167,81.291,1
597,PA,VISEU,58692.0,0.841,1.118,81.082,0
149,MA,CAXIAS,156973.0,0.751,0.008,80.979,0
262,MA,SANTA LUZIA,57635.0,0.751,0.000,80.979,0
60,AM,MANICORÉ,53914.0,0.736,1.000,80.556,0
445,MT,TABAPORÃ,9818.0,0.429,1.000,80.556,0
369,MT,ITIQUIRA,12236.0,0.397,1.000,80.556,1


## Discussão

- **A ordem das prioridades confere com o Cartografias 2024**: a proporção de municípios com facção, e em disputa, cresce de baixa para média e para alta.
- **O alerta antecipa parte da expansão de 2025.** Entre municípios que **não** tinham facção em 2024, a taxa de entrada em 2025 cresce com a prioridade calculada só com dados até 2023 (de cerca de 13% na baixa para cerca de 45% na alta). É uma validação fora do tempo, sem nenhum dado de 2025 no treino. O limiar rígido também separa bem, mas joga tudo em dois grupos; o fuzzy cria uma faixa intermediária que já concentra mais de 40% de entradas.
- **Cuidado ao interpretar as entradas.** O próprio Cartografias avisa que parte do aumento entre edições pode refletir a melhoria do método e mais operações policiais, e não só expansão real. 45 municípios "saíram" da lista entre 2024 e 2025, o que reforça esse alerta.
- **Interpretabilidade.** Cada regra é uma frase que um analista de inteligência entende e pode contestar.
- **Ética.** A saída é uma prioridade estatística para monitoramento, não uma acusação. O relatório não deve apresentar a lista de candidatos como municípios "dominados".

## Integração final do pipeline

VDE 2020–2023 + IBGE + Cartografias 2024 → **E1** árvore → **E2** regressão e logística → **E3** Bayes diagnóstico e transições → **E4** Naive Bayes + regimes ocultos (EM) → **E5** rede bayesiana: P(facção | porte, fronteira, sintomas) → **E6** fuzzy: prioridade de monitoramento → **validação com o Cartografias 2025**.